References:
- [Create Secret and IAM role for Federated](https://docs.aws.amazon.com/redshift/latest/dg/federated-create-secret-iam-role.html)
- [Federated Query Example](https://docs.aws.amazon.com/redshift/latest/dg/federated_query_example.html)
- [Getting Started with Federated Queries](https://docs.aws.amazon.com/redshift/latest/dg/getting-started-federated.html)
- [Getting started with Amazon Redshift Spectrum](https://docs.aws.amazon.com/redshift/latest/dg/c-getting-started-using-spectrum.html)

## Intro

Redshift Federated Queries - allows data in RDBMS systems (Postgres, Aurora, MySQL, SQL Server) to be access through Redshift.

Redshift Spectrum - query data in AWS S3 by exposing metadata using AWS Glue Data Catalog or Athena catalog.

To enable Federated Queries or Spectrum, Redshift needs to inherit permissions for the external system which can be done through IAM roles.

Create a new role: ITVRedshiftFederatedAndSpectrumDemoRole. Attach AmazonS3FullAccess policy to it. This role will be associated with the Redshift cluster later on.

## Setting up Postgres DB as source db

Set up the RDS Postgres DB and make sure the security group has the correct in bound permissions.
- admin: postgres
- pass: secret1234

Connect to the Postgres instance using psql then run the ff commands:

psql -h <endpoint> -p <port> -U <db_user> -W
```sql
CREATE DATABASE retail_db;

CREATE USER retail_user WITH ENCRYPTED PASSWORD 'itversity123';

GRANT ALL ON DATABASE retail_db TO retail_user;

\q
```

Connect to retail_db as postgres to grant priviledges on public schema to retail_user
```sql
GRANT ALL ON SCHEMA public TO retail_user;

GRANT USAGE ON SCHEMA public TO retail_user;

\q
```

Navigate to the retail_db_json project directory in your shell then..

Connect to the retail_db database as retail_user:

psql -h <endpoint> -p <port> -d retail_db -U retail_user -W
```sql

\i /home/ubuntu/retail_db_json/create_db_tables_pg.sql
-- tables should be created
```

#### Creating Secret using AWS Secrets Manager
Create retail1.secrets secret for the Postgres RDS instance, use retail_user as the user name.

## Using AWS Secrets with Python

Continue in secrets_manager.ipynb

## Create IAM policy for Secret and attach to Redshift Role and using Federated Queries

Create ITVRetailSecretPolicy, use arn of secret created earlier.

Attach the policy to the ITVRedshiftFederatedAndSpectrumDemoRole role.

Associate the role to the Redshift Cluster.

Before federated queries can be run, you need to create an external schema. Example:
```sql 
CREATE EXTERNAL SCHEMA retail_pg
FROM POSTGRES
DATABASE 'retail_db' SCHEMA 'public'
URI 'database-1.cjkkesmgkkv5.ap-southeast-2.rds.amazonaws.com'
IAM_ROLE 'arn:aws:iam::222634372385:role/ITVRedshiftFederatedAndSpectrumDemoRole'
SECRET_ARN 'arn:aws:secretsmanager:ap-southeast-2:222634372385:secret:retail1.secrets-QQPX0z';
```
Enable Enhanced VPC Routing on your Redshift Cluster and also enable publicly accessible routing.

Add Redshift Cluster security group to the inbound rules of the RDS Postgres security group for port 5432.

Try to query the Postgres RDS from the Redshift query editor.

### ERROR: authentication method 10 not supported

The error "authentication method 10 not supported" occurs because the Redshift federated query does not support the PostgreSQL scram-sha-256 authentication method (authentication method 10), which is the default in newer PostgreSQL versions (10+). Redshift currently requires the older md5 method. 

To resolve this, you must configure your PostgreSQL RDS instance to use md5 password encryption and update the user's password. This process requires modifying a parameter group and rebooting the RDS instance, which will cause a brief downtime. 

1. Create a Custom DB Parameter Group:
- Navigate to the Amazon RDS console and select Parameter groups.
- Create a new DB parameter group, ensuring it is compatible with your PostgreSQL version (e.g., aurora-postgresql14, postgres-13).
2. Modify the password_encryption Parameter:
- Edit the newly created parameter group.
- Search for the password_encryption parameter and change its value to md5.
- Save the changes.
3. Apply the Parameter Group to Your RDS Instance:
- Go to Databases and select your PostgreSQL RDS instance.
- Click Modify and change the DB parameter group to the new custom group you created.
- Choose Apply immediately to apply the changes right away.
4. Reboot the RDS Instance:
- Manually reboot the RDS instance for the parameter change to take full effect.
5. Re-encrypt the User Password to md5:
- Connect to your PostgreSQL database using a client tool like psql.
- Run the following SQL commands to ensure the connection uses md5 and then reset the password for the specific user used for the federated query. This step is crucial because existing passwords remain in their original encryption format until changed.
```sql
SET password_encryption = 'md5';
ALTER USER <your_username> WITH PASSWORD '<your_new_password>';
Replace <your_username> and <your_new_password> with your actual credentials. 
```
After these steps, your Amazon Redshift Serverless federated query should successfully authenticate with the PostgreSQL RDS instance.

## Query RDS using Federated Query and JOIN with local tables

```sql
SELECT c.customer_id,
    c.customer_fname,
    c.customer_lname,
    count(*)
FROM retail_pg.customers AS c
JOIN public.orders AS o
ON c.customer_id = o.order_customer_id
GROUP BY c.customer_id,
    c.customer_fname,
    c.customer_lname;
```

## Redshift Spectrum

#### Grant access on Glue Data Catalog to Redshift Cluster for Spectrum
Attach the AWSGlueConsoleFullAccess policy to ITVRedshiftFederatedAndSpectrumDemoRole role.  

Restore Redshift Cluster if needed, make sure ITVRedshiftFederatedAndSpectrumDemoRole role is associated with the cluster.

#### Create an external schema and an external table
```sql
-- create external schema from glue data catalog retail_db database
create external schema retail_spectrum 
from data catalog 
database 'retail_db' 
iam_role 'arn:aws:iam::222634372385:role/ITVRedshiftFederatedAndSpectrumDemoRole'
create external database if not exists;

SELECT COUNT(*) FROM retail_spectrum.orders;

SELECT * FROM retail_spectrum.orders LIMIT 10;
-- Redshift Spectrum will copy data from s3 into its cluster then process the data
```

#### Redshift Spectrum Uses Cases
- Data in S3 can be joined with data in Redshift tables to create a view or table that will be consumed by BI Tools (Tableau, PowerBI, etc)
```sql
-- public contains tables in redshift
-- retail_spectrum has data from s3 external with metadata from glue data catalog
SELECT c.customer_id,
    c.customer_fname,
    c.customer_lname,
    count(*)
FROM retail_spectrum.customers AS c
JOIN public.orders AS o
ON c.customer_id = o.order_customer_id
GROUP BY c.customer_id,
    c.customer_fname,
    c.customer_lname
ORDER BY count DESC;
```
